# RMSNorm — Paper-to-Code Mock (Colab)

**Paper:** Root Mean Square Layer Normalization (Zhang & Sennrich, 2019) — https://arxiv.org/abs/1910.07467

Mock (~30 min). Fill in the `RMSNorm` stub, run the invariance demo, then the sanity checks. Reference solution in the last cell.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0)

## 1. Implement RMSNorm (YOUR TASK)

`y = x / sqrt(mean(x^2) + eps) * g` over the last dim. No mean subtraction, no bias. Keep learnable gain `g` (init ones).

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        # TODO: self.g = nn.Parameter(...)  shape (dim,), init ones
        raise NotImplementedError("fill me in")

    def forward(self, x):
        # TODO: rms = sqrt(mean(x^2, last dim) + eps); return x / rms * g
        raise NotImplementedError("fill me in")

## 2. Demonstrate the defining property (scale vs shift)
Scale invariance is the contribution; lack of shift invariance is the simplification vs LayerNorm.

In [ ]:
norm = RMSNorm(dim=8)
x = torch.randn(4, 8)
y = norm(x)
print("max |RMSNorm(7.5x) - RMSNorm(x)| =", (norm(7.5 * x) - y).abs().max().item(), "(~0)")
print("max |RMSNorm(x+3) - RMSNorm(x)| =", (norm(x + 3.0) - y).abs().max().item(), "(not ~0)")

## 3. Sanity checks

In [ ]:
# 1) g=1 -> each row has unit RMS
norm = RMSNorm(16); x = torch.randn(32, 16) * 5.0
row_rms = norm(x).pow(2).mean(dim=-1).sqrt()
assert torch.allclose(row_rms, torch.ones_like(row_rms), atol=1e-3)

# 2) scale invariant
x = torch.randn(4, 16)
assert torch.allclose(norm(x), norm(123.4 * x), atol=1e-5)

# 3) NOT shift invariant
assert not torch.allclose(norm(x), norm(x + 2.0), atol=1e-3)

# 4) shape preserved; only 'g' is learnable (no bias)
assert norm(x).shape == x.shape
assert [n for n, p in norm.named_parameters() if p.requires_grad] == ['g']

# 5) gradient reaches g
norm = RMSNorm(8); norm(torch.randn(5, 8)).pow(2).sum().backward()
assert norm.g.grad is not None and norm.g.grad.abs().sum() > 0

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class RMSNormSolution(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.g = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.float().pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return (x.float() / rms).type_as(x) * self.g